# L8b: Two Component Systems
In this lecture, we'll explore the two-component systems (TCSs) in bacteria. These systems are a type of signal transduction pathway that allows bacteria to _sense and respond_ to changes in their environment. The key ideas of the lecture are:

* TCSs are composed of two proteins: a sensor kinase and a response regulator. The sensor kinase detects a specific environmental signal and phosphorylates the response regulator, which then carries out the appropriate cellular response. 
* TCSs play a critical role in bacterial adaptation to changing conditions and are involved in a wide range of cellular processes, including virulence, motility, and biofilm formation. 
* The PhoR/PhoB TCS is a well-studied example of a TCS that regulates the response to phosphate limitation in bacteria. PhoR is the sensor kinase that detects low phosphate levels, while PhoB is the response regulator that activates genes involved in phosphate acquisition and metabolism.

## Rewiew: Carbon Catabolite Repression in Bacteria
In the previous lecture, we discussed the concept of carbon catabolite repression in bacteria. This regulatory mechanism allows bacteria to preferentially use a specific carbon source when multiple carbon sources are available.  

Let's review the key points of carbon catabolite repression in _Bacillus subtilis_ by exploring the publication:
* [Singh KD, Schmalisch MH, Stülke J, Görke B. Carbon catabolite repression in Bacillus subtilis: quantitative analysis of repression exerted by different carbon sources. J Bacteriol. 2008 Nov;190(21):7275-84. doi: 10.1128/JB.00848-08. Epub 2008 Aug 29. PMID: 18757537; PMCID: PMC2580719.](https://pubmed.ncbi.nlm.nih.gov/18757537/)

Understanding the mechanisms of carbon catabolite repression is essential for understanding how bacteria regulate their metabolism in response to changing environmental conditions. However, we can also use the _parts_ that make up these regulatory networks to build more complex systems:

* [Guo S, Du J, Li D, Xiong J, Chen Y. Versatile xylose and arabinose genetic switches development for yeasts. Metab Eng. 2025 Jan;87:21-36. doi: 10.1016/j.ymben.2024.11.004. Epub 2024 Nov 12. PMID: 39537022.](https://pubmed.ncbi.nlm.nih.gov/39537022/)

## Two Component Systems
Two-component systems (TCSs) are composed of two proteins: a sensor kinase and a response regulator. The sensor kinase detects a specific environmental signal and phosphorylates the response regulator, which then carries out the appropriate cellular response.

Let's review this publication to understand the structure and function of TCSs in bacteria:
* [Jacob-Dubuisson F, Mechaly A, Betton JM, Antoine R. Structural insights into the signalling mechanisms of two-component systems. Nat Rev Microbiol. 2018 Oct;16(10):585-593. doi: 10.1038/s41579-018-0055-7. PMID: 30008469.](https://pubmed.ncbi.nlm.nih.gov/30008469/)

## Problem set 3 (PS3): PhoR/PhoB Two-Component System
The PhoR/PhoB two-component system is a well-studied example of a TCS that regulates the response to phosphate limitation in bacteria. PhoR is the sensor kinase that detects low phosphate levels, while PhoB is the response regulator that activates genes involved in phosphate acquisition and metabolism. 

Review of the PhoR/PhoB TCS in _Escherichia coli_:
* [Gardner SG, McCleary WR. Control of the phoBR Regulon in Escherichia coli. EcoSal Plus. 2019 Sep;8(2):10.1128/ecosalplus.ESP-0006-2019. doi: 10.1128/ecosalplus.ESP-0006-2019. PMID: 31520469; PMCID: PMC11573284.](https://pubmed.ncbi.nlm.nih.gov/31520469/)

### Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants. 
* The `Include.jl` file also loads external packages, various functions that we will use in the exercise, and custom types to model the components of our problem. It checks for a `Manifest.toml` file; if it finds one, packages are loaded. Other packages are downloaded and then loaded.

In [4]:
include("Include.jl");

__Build the model__. To store all the problem data, we created [the `MyPrimalFluxBalanceAnalysisCalculationModel` type](src/Types.jl). Let's build one of these objects for our problem and store it in the `model::MyPrimalFluxBalanceAnalysisCalculationModel` variable. We also return the `rd::Dict{String, String}` dictionary, which maps the reaction name field (key) to the reaction string (value).
* __Builder (or factory) pattern__: For all custom types that we make, we'll use something like [the builder software pattern](https://en.wikipedia.org/wiki/Builder_pattern) to construct and initialize these objects. The calling syntax will be the same for all types: [a `build(...)` method](src/Factory.jl) will take the kind of thing we want to build in the first argument, and the data needed to make that type as [a `NamedTuple` instance](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) in the second argument.
* __What's the story with the `let` block__? A [let block](https://docs.julialang.org/en/v1/manual/variables-and-scoping/#Let-Blocks) creates a new hard scope and new variable bindings each time they run. Thus, they act like a private scratch space, where data comes in (is captured by the block), but only what we want to be exposed comes out. 

In [5]:
model, rd = let

    # first, load the reaction file - and process it
    listofreactions = read_reaction_file(joinpath(_PATH_TO_DATA, "PHOB-3-5-2011.net")); # load the reactions from the VFF reaction file
    S, species, reactions, rd = build_stoichiometric_matrix(listofreactions); # Builds the stochiometric matrix, species list, and the reactions list
    boundsarray = build_default_bounds_array(listofreactions); # Builds a default bounds model using the flat file flags

    # build the FBA model -
    model = build(MyPrimalFluxBalanceAnalysisCalculationModel, (
        S = S, # stoichiometric matrix
        fluxbounds = boundsarray, # these are the *default* bounds, we'll need to update with new info if we have it
        species = species, # list of species. The rows of S are in this order
        reactions = reactions, # list of reactions. The cols of S are in this order
        objective = length(reactions) |> R -> zeros(R), # this is empty, we'll need to set this
    ));

    # return -
    model, rd
end;

In [7]:
rd

Dict{String, String} with 70 entries:
  "DEGRADATION"                  => "DEGRADATION,PI_PERIPLASIM_PhoR_MEMBRANE,[]…
  "UGPB_TF_RNAP"                 => "UGPB_TF_RNAP,G_UGPB_P_PhoB+RNAP,G_UGPB_P_P…
  "PHOS_DEGRADATION"             => "PHOS_DEGRADATION,PhoS,[],0,inf;"
  "PORIN_MEMBRANE_DEGRADATION"   => "PORIN_MEMBRANE_DEGRADATION,PORIN_MEMBRANE,…
  "PHOA_TF_START"                => "PHOA_TF_START,G_PHOA_P_PhoB_RNAP,G_PHOA+P_…
  "PORIN_TRANSLOCATION_MEMBRANE" => "PORIN_TRANSLOCATION_MEMBRANE,PORIN,PORIN_M…
  "RIBOSOME_ASSEMBLY"            => "RIBOSOME_ASSEMBLY,[],RIBOSOME,0,inf;"
  "PI_TRANSPORT"                 => "PI_TRANSPORT,PORIN_MEMBRANE_PI_BOUNDARY,PO…
  "BIND_PhoB_PPASE"              => "BIND_PhoB_PPASE,P_PhoB+PhoB_PPASE_ADP,P_Ph…
  "ADP_SYNTHESIS"                => "ADP_SYNTHESIS,[],ADP,0,inf;"
  "PHOA_START"                   => "PHOA_START,G_PHOA_RNAP,G_PHOA+RNAP+mRNA_PH…
  "RNAP_ASSEMBLY"                => "RNAP_ASSEMBLY,[],RNAP,0,inf;"
  "ACTIVATE_PhoB_BIND"           => 

In [9]:
model.fluxbounds

73×2 Matrix{Float64}:
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
     ⋮    
     0.0  1000.0
 -1000.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0
 -1000.0  1000.0
     0.0  1000.0
     0.0  1000.0
     0.0  1000.0